In [1]:
# Importing the libraries

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Create a dataset
np.random.seed(42)
num_samples = 500

data = {
    "house_size": np.random.normal(2000, 500, num_samples),             # sqft
    "num_rooms": np.random.randint(3, 8, num_samples),                  # rooms
    "crime_rate": np.random.uniform(0, 1, num_samples),                 # 0 = safe, 1 = dangerous
    "distance_to_city_center": np.random.uniform(1, 30, num_samples),   # miles
    "school_rating": np.random.uniform(1, 10, num_samples),             # 1-10
    "age_of_house": np.random.randint(1, 80, num_samples),              # years
}

df = pd.DataFrame(data)

# Target variable: House Price
df["price"] = (
    df["house_size"] * 150
    + df["school_rating"] * 4000
    - df["crime_rate"] * 30000
    - df["age_of_house"] * 500
    - df["distance_to_city_center"] * 1500
    + np.random.normal(0, 20000, num_samples)  # noise
)
df.head()

,house_size,num_rooms,crime_rate,distance_to_city_center,school_rating,age_of_house,price
0,2248.357077,4,0.768273,23.635240,8.257633,60,251425.554434
1,1930.867849,4,0.417767,8.969375,5.133159,66,244035.162978
2,2323.844269,5,0.421357,24.855815,1.467609,66,289330.307496
3,2761.514928,5,0.737582,13.288409,8.076504,13,409477.271649
4,1882.923313,7,0.238777,20.358947,2.812274,56,237999.172488


In [3]:
X = df.drop("price", axis=1)
y = df["price"]

In [4]:
# 2- Add Constant for intercept (required for statsmodels OLS)
X_sm = sm.add_constant(X)

In [5]:
X_sm

,const,house_size,num_rooms,crime_rate,distance_to_city_center,school_rating,age_of_house
0,1.0,2248.357077,4,0.768273,23.635240,8.257633,60
1,1.0,1930.867849,4,0.417767,8.969375,5.133159,66
2,1.0,2323.844269,5,0.421357,24.855815,1.467609,66
3,1.0,2761.514928,5,0.737582,13.288409,8.076504,13
4,1.0,1882.923313,7,0.238777,20.358947,2.812274,56
...,...,...,...,...,...,...,...
495,1.0,2269.455022,7,0.411028,12.941201,3.696091,15
496,1.0,1481.376923,6,0.839861,19.878231,1.691759,32
497,1.0,1904.830661,3,0.900023,16.817524,5.505618,68
498,1.0,1562.190873,6,0.353421,2.805920,8.150640,18


### Perform Backward Feature Elimination

In [6]:
def back_elimination(X, y, significance_level=0.05):
    X_modeled = X.copy()
    while True:
        model = sm.OLS(y, X_modeled).fit()
        p_values = model.pvalues
        max_p_value = p_values.max()
        if max_p_value > significance_level:
            excluded_feature = p_values.idxmax()
            print(f"Removing {excluded_feature} with p-value {max_p_value}")
            X_modeled = X_modeled.drop(columns=[excluded_feature])
        else:
            break
    return X_modeled, model

In [8]:
X_selected, final_model = back_elimination(X_sm, y)

Removing const with p-value 0.21578450717368222
Removing num_rooms with p-value 0.4812683869791907


In [9]:
print("\nSelected Features after Backward Elimination:")
print(X_selected.columns)


Selected Features after Backward Elimination:
Index(['house_size', 'crime_rate', 'distance_to_city_center', 'school_rating',
       'age_of_house'],
      dtype='object')


In [10]:
print("\nFinal Model Summary:")
print(final_model.summary())


Final Model Summary:
                                 OLS Regression Results                                
Dep. Variable:                  price   R-squared (uncentered):                   0.995
Model:                            OLS   Adj. R-squared (uncentered):              0.995
Method:                 Least Squares   F-statistic:                          2.036e+04
Date:                Thu, 27 Nov 2025   Prob (F-statistic):                        0.00
Time:                        10:50:32   Log-Likelihood:                         -5641.2
No. Observations:                 500   AIC:                                  1.129e+04
Df Residuals:                     495   BIC:                                  1.131e+04
Df Model:                           5                                                  
Covariance Type:            nonrobust                                                  
                              coef    std err          t      P>|t|      [0.025      0.975]
------

In [11]:
# 4. Train a simple Linear Regression model on selected features
# -----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

In [12]:
# 5. Evaluate the model
# -----------------------------------------------------
print("\nModel R2 Score:", r2_score(y_test, y_pred))
print("Model RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


Model R2 Score: 0.9395648590050748
Model RMSE: 19284.86009951475
